In [ ]:
# ============================================
# BRONZE LAYER - DATA INGESTION
# ============================================

import logging
import requests
from pyspark.sql.functions import current_timestamp
from configs.schema import sales_schema, exchange_schema

# ============================================
# CONFIGURATION
# ============================================

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ============================================
# DOWNLOAD FUNCTION
# ============================================

def download_file(url: str, output_path: str):
    logger.info(f"Downloading: {url}")
    r = requests.get(url)
    with open(f"/lakehouse/default/{output_path}", "wb") as f:
        f.write(r.content)
    logger.info(f"Saved to: {output_path}")

# ============================================
# INGEST FUNCTION
# ============================================

def ingest_to_bronze(file_path: str, schema, output_path: str):
    logger.info(f"Processing: {file_path}")

    df = spark.read.format("csv") \
        .option("header", True) \
        .option("mode", "FAILFAST") \
        .schema(schema) \
        .load(file_path)

    df = df.withColumn("ingestion_time", current_timestamp())

    df.write.mode("overwrite").parquet(output_path)

    logger.info(f"Written: {output_path}")

# ============================================
# MAIN PIPELINE
# ============================================

def main():

    sources = [
        {
            "url": "https://drive.google.com/uc?export=download&id=13Ng8D79D6NxWQ3CD4eE8CAKpsvoDmVue",
            "schema": sales_schema,
            "raw_path": "Files/Bronze/sales.csv",
            "out_path": "Files/Bronze/processed/sales_raw"
        },
        {
            "url": "https://drive.google.com/uc?export=download&id=1nVMEuCXbzPWd25UNrvC-y02mWDPWVf1W",
            "schema": exchange_schema,
            "raw_path": "Files/Bronze/exchange_rate.csv",
            "out_path": "Files/Bronze/processed/exchange_raw"
        }
    ]

    for src in sources:
        download_file(src["url"], src["raw_path"])
        ingest_to_bronze(src["raw_path"], src["schema"], src["out_path"])

    logger.info("Bronze ingestion completed")

# ============================================
# EXECUTION
# ============================================

main()
mssparkutils.notebook.exit("Success")